# 02 — EDA: Electricity Prices

Explore the DE_LU day-ahead price series:
- Seasonality (hour-of-day, day-of-week, month)
- Distribution analysis and outliers (negative prices)
- Autocorrelation (ACF/PACF)
- Correlation with load and weather

**Prerequisites:** Bronze layer built (`python -m etl.bronze`).

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

con = duckdb.connect()
PRICES = '../data/bronze/prices.parquet'
LOAD   = '../data/bronze/load.parquet'
WEATHER = '../data/bronze/weather.parquet'

## 2.1 Full price time series

In [ ]:
df = con.execute(f"SELECT * FROM read_parquet('{PRICES}') ORDER BY timestamp_utc").fetchdf()
print(f'Shape: {df.shape}')
print(df.describe())

fig = px.line(df, x='timestamp_utc', y='price_eur_mwh',
              title='DE_LU Day-Ahead Price — Full History', template='plotly_dark')
fig.update_layout(xaxis_title='', yaxis_title='EUR/MWh', height=400)
fig.show()

## 2.2 Negative prices (a key electricity market phenomenon)

In [ ]:
df_neg = con.execute(f"""
    SELECT
        EXTRACT(YEAR FROM timestamp_utc) AS year,
        COUNT(*) AS n_negative_hours,
        AVG(price_eur_mwh) AS avg_negative_price
    FROM read_parquet('{PRICES}')
    WHERE price_eur_mwh < 0
    GROUP BY 1 ORDER BY 1
""").fetchdf()
print('Negative hours by year:')
print(df_neg)

fig = px.bar(df_neg, x='year', y='n_negative_hours',
             title='Negative Price Hours per Year', template='plotly_dark')
fig.show()

## 2.3 Average price by hour of day and day of week

In [ ]:
df_heatmap = con.execute(f"""
    SELECT
        EXTRACT(DOW  FROM timestamp_utc) AS day_of_week,
        EXTRACT(HOUR FROM timestamp_utc) AS hour_of_day,
        AVG(price_eur_mwh) AS avg_price
    FROM read_parquet('{PRICES}')
    GROUP BY 1, 2 ORDER BY 1, 2
""").fetchdf()

pivot = df_heatmap.pivot(index='day_of_week', columns='hour_of_day', values='avg_price')
dow_labels = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

fig = px.imshow(pivot, labels=dict(x='Hour (UTC)', y='Day of week', color='EUR/MWh'),
                y=dow_labels, title='Average Price Heatmap (DoW × Hour)',
                color_continuous_scale='RdYlGn_r', template='plotly_dark')
fig.show()

## 2.4 Autocorrelation at key lags

In [ ]:
# Manual ACF via DuckDB LAG + CORR
lags = [1, 2, 3, 6, 12, 24, 48, 72, 96, 120, 144, 168]
acf_vals = []
for lag in lags:
    r = con.execute(f"""
        SELECT CORR(price_eur_mwh, LAG(price_eur_mwh, {lag}) OVER (ORDER BY timestamp_utc))
        FROM read_parquet('{PRICES}')
    """).fetchone()[0]
    acf_vals.append(r)

fig = go.Figure(go.Bar(x=[str(l) for l in lags], y=acf_vals,
                       marker_color=['#EF553B' if v < 0 else '#636EFA' for v in acf_vals]))
fig.update_layout(title='Autocorrelation by Lag (hours) — strong 24h and 168h peaks expected',
                  xaxis_title='Lag (hours)', yaxis_title='Pearson r',
                  template='plotly_dark', height=350)
fig.show()

## 2.5 Price vs load and temperature

In [ ]:
df_corr = con.execute(f"""
    SELECT
        CORR(p.price_eur_mwh, l.load_mw)        AS price_load_corr,
        CORR(p.price_eur_mwh, w.temperature_2m)  AS price_temp_corr,
        CORR(p.price_eur_mwh, w.solar_radiation) AS price_solar_corr,
        CORR(p.price_eur_mwh, w.wind_speed_10m)  AS price_wind_corr
    FROM read_parquet('{PRICES}') p
    JOIN read_parquet('{LOAD}') l USING (timestamp_utc)
    JOIN read_parquet('{WEATHER}') w USING (timestamp_utc)
""").fetchdf()
print('Pearson correlations with price:')
print(df_corr.T.rename(columns={0: 'correlation'}).round(4))